In [3]:
from google.colab import files

uploaded = files.upload()

Saving Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.csv to Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.csv


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv('Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.csv')
print(df)
print("=" * 65)
print("STEP 1 — Raw Data Shape:", df.shape)
print("Columns:", df.columns.tolist())
print()

                Timestamp How old are you?  What industry do you work in?  \
0      4/27/2021 11:02:10            25-34   Education (Higher Education)   
1      4/27/2021 11:02:22            25-34              Computing or Tech   
2      4/27/2021 11:02:38            25-34  Accounting, Banking & Finance   
3      4/27/2021 11:02:41            25-34                     Nonprofits   
4      4/27/2021 11:02:42            25-34  Accounting, Banking & Finance   
...                   ...              ...                            ...   
28202  2/24/2026 15:34:11            18-24        Agriculture or Forestry   
28203  2/24/2026 15:39:58            18-24                  Entertainment   
28204  2/24/2026 15:44:23            18-24                            Law   
28205  2/25/2026 13:04:14            18-24   Engineering or Manufacturing   
28206  2/28/2026 19:41:34         under 18                            NaN   

                                      Job title  \
0            Research an

upload and read data

In [5]:
salary_data = df.rename(columns={
    "Timestamp":                                                     "timestamp",
    "How old are you?":                                              "age_group",
    "What industry do you work in?":                                 "industry",
    "Job title":                                                     "job_title",
    "If your job title needs additional context, please clarify here:": "job_context",
    "What is your annual salary? (You'll indicate the currency in a later question. "
    "If you are part-time or hourly, please enter an annualized equivalent -- what you "
    "would earn if you worked the job 40 hours a week, 52 weeks a year.)":             "salary_data",
    "How much additional monetary compensation do you get, if any (for example, bonuses "
    "or overtime in an average year)? Please only include monetary compensation here, "
    "not the value of benefits.":                                    "bonus_data",
    "Please indicate the currency":                                  "currency",
    "If \"Other,\" please indicate the currency here: ":             "currency_other",
    "If your income needs additional context, please provide it here:": "income_context",
    "What country do you work in?":                                  "country",
    "If you're in the U.S., what state do you work in?":             "us_state",
    "What city do you work in?":                                     "city",
    "How many years of professional work experience do you have overall?": "exp_overall",
    "How many years of professional work experience do you have in your field?": "exp_field",
    "What is your highest level of education completed?":            "education",
    "What is your gender?":                                          "gender",
    "What is your race? (Choose all that apply.)":                   "race",
})
print("STEP 2 — Columns renamed.")
print(salary_data)

STEP 2 — Columns renamed.
                timestamp age_group                       industry  \
0      4/27/2021 11:02:10     25-34   Education (Higher Education)   
1      4/27/2021 11:02:22     25-34              Computing or Tech   
2      4/27/2021 11:02:38     25-34  Accounting, Banking & Finance   
3      4/27/2021 11:02:41     25-34                     Nonprofits   
4      4/27/2021 11:02:42     25-34  Accounting, Banking & Finance   
...                   ...       ...                            ...   
28202  2/24/2026 15:34:11     18-24        Agriculture or Forestry   
28203  2/24/2026 15:39:58     18-24                  Entertainment   
28204  2/24/2026 15:44:23     18-24                            Law   
28205  2/25/2026 13:04:14     18-24   Engineering or Manufacturing   
28206  2/28/2026 19:41:34  under 18                            NaN   

                                      job_title  job_context salary_data  \
0            Research and Instruction Librarian          

rename columns

In [8]:
def clean_currency_col(series):
    """Strip commas/spaces and cast to float."""
    return (
        series.astype(str)
              .str.replace(r"[,$\s]", "", regex=True)
              .str.extract(r"([\d.]+)")[0]
              .pipe(pd.to_numeric, errors="coerce")
    )


create function to make data easier to read (keep values as text, remove symbols, convert text into numbers). Anything that can't be read will have error msg.

In [9]:
dataframe1 = (
    salary_data
    .assign(salary=clean_currency_col(salary_data["salary_data"]))
    .assign(bonus=clean_currency_col(salary_data["bonus_data"]))
    .assign(total_comp=lambda d: d["salary"].add(d["bonus"].fillna(0)))
)

print("STEP 3 — Salary/bonus parsed.  Sample totals (first 5):")
print(dataframe1[["job_title", "salary", "bonus", "total_comp"]].head())
print()

STEP 3 — Salary/bonus parsed.  Sample totals (first 5):
                                  job_title  salary   bonus  total_comp
0        Research and Instruction Librarian   55000     0.0     55000.0
1  Change & Internal Communications Manager   54600  4000.0     58600.0
2                      Marketing Specialist   34000     NaN     34000.0
3                           Program Manager   62000  3000.0     65000.0
4                        Accounting Manager   60000  7000.0     67000.0



clean columns and calculate salaries with lambda.

In [10]:
col_map = {
    'What industry do you work in?': 'industry',
    "What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)": 'annual_salary',
    'How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.': 'bonus',
    'Please indicate the currency': 'currency',
    'How many years of professional work experience do you have overall?': 'exp_total'
}
df = df.rename(columns=col_map)

rename columns

In [11]:
df['annual_salary'] = pd.to_numeric(df['annual_salary'].astype(str).str.replace(',', ''), errors='coerce')
df['bonus'] = pd.to_numeric(df['bonus'], errors='coerce').fillna(0)

df['total_comp'] = df['annual_salary'] + df['bonus']

df_usd = df[df['currency'] == 'USD'].dropna(subset=['annual_salary', 'industry'])

df_long = pd.melt(df_usd,
                  id_vars=['industry', 'exp_total', 'total_comp'],
                  value_vars=['annual_salary', 'bonus'],
                  var_name='comp_type',
                  value_name='amount')

clean columns

In [12]:
print("--- TIDY DATA PREVIEW (Long Format) ---")
print(df_long.head(10))

print("\n--- DATA INFO ---")
print(df_long.info())

--- TIDY DATA PREVIEW (Long Format) ---
                        industry      exp_total  total_comp      comp_type  \
0   Education (Higher Education)      5-7 years     55000.0  annual_salary   
1  Accounting, Banking & Finance    2 - 4 years     34000.0  annual_salary   
2                     Nonprofits   8 - 10 years     65000.0  annual_salary   
3  Accounting, Banking & Finance   8 - 10 years     67000.0  annual_salary   
4   Education (Higher Education)   8 - 10 years     62000.0  annual_salary   
5                     Publishing    2 - 4 years     35000.0  annual_salary   
6  Education (Primary/Secondary)      5-7 years     50000.0  annual_salary   
7              Computing or Tech  21 - 30 years    122000.0  annual_salary   
8  Accounting, Banking & Finance  21 - 30 years     45000.0  annual_salary   
9                     Nonprofits      5-7 years     47500.0  annual_salary   

     amount  
0   55000.0  
1   34000.0  
2   62000.0  
3   60000.0  
4   62000.0  
5   33000.0  
6  

print tidy data